### least square regression optimization using GD & SGD

In [1]:
import numpy as np

## synthetic  data

features_len = 5
samples_len = 1000

true_weights = np.arange(1,1+features_len)

points = []
for i in range(samples_len):
    x = np.random.randn(features_len)
    y = x.dot(true_weights) + np.random.randn()
    points.append((x,y))

In [2]:
points[0]

(array([ 0.77998124,  1.62892531,  1.0650634 ,  1.38865241, -0.30375079]),
 np.float64(11.757763949690142))

In [3]:
def mean_squared_error(w):

    return  sum([(w.dot(x)-y)**2 for x,y in points]) / samples_len

def mean_squared_error_gradient(w):

    return  sum([ 2*(w.dot(x)-y)*x  for x,y in points ]) / samples_len


def mean_squared_error_sgd(w,i):

    X,y = points[i]
    
    return ( w.dot(X) - y )**2


def mean_squared_error_gradient_sgd(w,i):

    X,y = points[i]
    
    return 2 * (w.dot(X) - y) * X



In [4]:
import time

def track_time(fn):

    def wrapper(*args,**kwargs):

        start_time = time.time()

        results = fn(*args,**kwargs)
        
        end_time = time.time()-start_time

        print('Total Time Taken : ',end_time)

        return results

    return wrapper



@track_time
def gradient_descent(n, mean_squared_fn, mean_squared_grad):

    w = np.zeros(features_len)
    eta = 0.001
    patience = 0

    best_loss = float('inf')
    loss = 0
    grad = 0

    for i in range(n):

        loss = mean_squared_fn(w)
        grad = mean_squared_grad(w)

        w -= eta*grad

        if loss < best_loss:
            best_loss = loss

        else:
            patience +=1

        if patience == 10:
            break

        # print(f'Iteration : [{i+1}/{n}] : Weights : {w} | Loss : {loss} ')
    
    print('-------- Summary --------')
    print('No of epochs taken to converge : ', i)
    print('Final loss : ', loss)
    print('Final weights : ',w)
    print('---------------------------')

    return w, loss

final_weights, final_loss = gradient_descent(10000,mean_squared_error,mean_squared_error_gradient)


-------- Summary --------
No of epochs taken to converge :  9010
Final loss :  0.9917039036413399
Final weights :  [0.97857946 1.96782941 3.01520598 4.01224743 4.95459451]
---------------------------
Total Time Taken :  13.494630575180054


In [5]:
## Stochastic Gradient Descent 

    
@track_time
def stochastic_gradient_descent(n, mean_squared_fn, mean_squared_grad):

    w = np.zeros(features_len)
    eta = 1
    patience = 0

    best_loss = float('inf')
    loss = 0
    grad = 0
    num_updates = 1.0

    m = np.random.randint(len(points)//2)

    for i in range(n):

        for j in range(m):
            loss = mean_squared_fn(w,j)
            grad = mean_squared_grad(w,j)
            w -= eta*grad
            
            eta = 1.0 / num_updates
            num_updates +=1


            if loss < best_loss:
                best_loss = loss

            else:
                patience +=1

            if patience == 10:
                break

        # print(f'Iteration : [{i+1}/{n}] : Weights : {w} | Loss : {loss} ')
    
    print('-------- Summary --------')
    print('No of epochs taken to converge : ', i)
    print('Final loss : ', loss)
    print('Final weights : ',w)
    print('---------------------------')

    return w, loss

final_weights, final_loss = stochastic_gradient_descent(10000,mean_squared_error_sgd,mean_squared_error_gradient_sgd)


-------- Summary --------
No of epochs taken to converge :  9999
Final loss :  4.216835984374329
Final weights :  [0.93768104 1.95322921 3.04741998 4.00058611 5.01221746]
---------------------------
Total Time Taken :  8.384789943695068


In [6]:
# SGD is faster than GD.